In [2]:
import rdflib
from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDF, RDFS, XSD, OWL
import networkx as nx

In [3]:
nkb = Graph()
nkb.parse('./mappings/NKB_RDF_V3.ttl')

<Graph identifier=N936949e7238444d18f18e77048e10153 (<class 'rdflib.graph.Graph'>)>

In [4]:
def build_efficient_reference_graph(g):
    """
    Build a graph from RDF data efficiently using indexing rather than exhaustive search.
    """
    import rdflib
    import networkx as nx
    from collections import defaultdict
    
    # Create a NetworkX graph
    nx_graph = nx.DiGraph()
    
    # Define entity types with their IRIs
    entity_iris = {
        'material': 'http://purl.bioontology.org/ontology/npo#NPO_199',
        'assay': 'http://purl.obolibrary.org/obo/OBI_0000070',
        'publication': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C48471',
        'medium': 'http://purl.bioontology.org/ontology/npo#NPO_1853',
        'result': 'http://www.bioassayontology.org/bao#BAO_0000179',
        'parameter': 'http://purl.bioontology.org/ontology/npo#NPO_1680',
        'materialfg': 'http://purl.bioontology.org/ontology/npo#NPO_174',
        'additive': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C63495',
        'contam': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C84280',
        'molecularresult': 'http://purl.obolibrary.org/obo/ERO_0000833'
    }
    
    # Add all entities as nodes to the graph
    print("Adding entities as nodes...")
    entities = {}
    for entity_name, entity_iri in entity_iris.items():
        for s, p, o in g.triples((None, rdflib.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), 
                                  rdflib.URIRef(entity_iri))):
            if not isinstance(s, rdflib.BNode):
                str_s = str(s)
                entities[str_s] = {'type': entity_name}
                nx_graph.add_node(str_s, type=entity_name)
    
    print(f"Added {len(entities)} entity nodes to the graph")
    
    # Create indexes for efficient lookups
    # 1. Index of entities by their primary ID
    id_to_entity = defaultdict(dict)
    # 2. Index of entities by DOI
    doi_to_entities = defaultdict(list)
    
    # Define key predicates
    doi_predicate = rdflib.URIRef('http://purl.obolibrary.org/obo/OBI_0002110')
    id_predicate = rdflib.URIRef('http://purl.org/dc/terms/identifier')
    
    # First pass: build indexes
    print("Building indexes...")
    for entity_uri_str, entity_data in entities.items():
        entity_uri = rdflib.URIRef(entity_uri_str)
        entity_type = entity_data['type']
        
        # Find direct IDs and DOIs
        for s, p, o in g.triples((entity_uri, id_predicate, None)):
            if isinstance(o, rdflib.Literal):
                id_value = str(o)
                id_to_entity[entity_type][id_value] = entity_uri_str
        
        for s, p, o in g.triples((entity_uri, doi_predicate, None)):
            if isinstance(o, rdflib.Literal):
                doi_value = str(o)
                doi_to_entities[doi_value].append((entity_uri_str, entity_type))
    
    # Second pass: find IDs and DOIs in BNodes
    print("Finding references through BNodes...")
    
    # Track BNode connection types for statistics
    bnode_connection_types = defaultdict(int)
    
    for entity_uri_str, entity_data in entities.items():
        entity_uri = rdflib.URIRef(entity_uri_str)
        entity_type = entity_data['type']
        
        # Look for BNodes connected to this entity
        for s, link_predicate, o in g.triples((entity_uri, None, None)):
            if isinstance(o, rdflib.BNode):
                bnode = o
                link_type = str(link_predicate)
                
                # Check for IDs in the BNode
                for bs, bp, bo in g.triples((bnode, id_predicate, None)):
                    if isinstance(bo, rdflib.Literal):
                        id_value = str(bo)
                        
                        # Determine the target type based on the link predicate
                        target_type = None
                        if link_predicate == rdflib.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C93400'):
                            target_type = 'material'
                        elif link_predicate == rdflib.URIRef('http://purl.bioontology.org/ontology/npo#NPO_1853'):
                            target_type = 'medium'
                        elif link_predicate == rdflib.URIRef('http://purl.obolibrary.org/obo/OBI_0000070'):
                            target_type = 'assay'
                        
                        if target_type and id_value in id_to_entity[target_type]:
                            target_uri_str = id_to_entity[target_type][id_value]
                            nx_graph.add_edge(entity_uri_str, target_uri_str, 
                                             relation=f"references_{target_type}",
                                             id=id_value)
                            bnode_connection_types[f"{entity_type}_to_{target_type}"] += 1
                
                # Check for DOIs in the BNode
                for bs, bp, bo in g.triples((bnode, doi_predicate, None)):
                    if isinstance(bo, rdflib.Literal):
                        doi_value = str(bo)
                        
                        # Find entities with this DOI
                        if doi_value in doi_to_entities:
                            for target_uri_str, target_type in doi_to_entities[doi_value]:
                                if entity_uri_str != target_uri_str:  # Avoid self-loops
                                    nx_graph.add_edge(entity_uri_str, target_uri_str, 
                                                     relation=f"shared_doi_with_{target_type}",
                                                     doi=doi_value)
                                    bnode_connection_types[f"{entity_type}_to_{target_type}_doi"] += 1
    
    print("BNode connection types found:")
    for conn_type, count in sorted(bnode_connection_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  {conn_type}: {count}")
    
    # Look for direct column references
    print("Finding direct column references...")
    column_references = 0
    
    for entity_uri_str, entity_data in entities.items():
        entity_uri = rdflib.URIRef(entity_uri_str)
        
        # Look for literals that represent column references
        for column_name in ['material_MaterialID', 'assay_AssayID', 'medium_MediumID', 'publication_DOI']:
            for s, p, o in g.triples((entity_uri, rdflib.Literal(column_name), None)):
                if isinstance(o, rdflib.Literal):
                    ref_value = str(o)
                    ref_type = column_name.split('_')[0]  # 'material' from 'material_MaterialID'
                    
                    if ref_type in id_to_entity and ref_value in id_to_entity[ref_type]:
                        target_uri_str = id_to_entity[ref_type][ref_value]
                        nx_graph.add_edge(entity_uri_str, target_uri_str, 
                                         relation=f"direct_ref_to_{ref_type}",
                                         value=ref_value)
                        column_references += 1
    
    print(f"Found {column_references} direct column references")
    
    # Print final statistics
    print(f"Created graph with {nx_graph.number_of_nodes()} nodes and {nx_graph.number_of_edges()} edges")
    
    # Calculate edge types
    edge_types = defaultdict(int)
    for s, t, data in nx_graph.edges(data=True):
        source_type = nx_graph.nodes[s].get('type', 'unknown')
        target_type = nx_graph.nodes[t].get('type', 'unknown')
        edge_types[(source_type, target_type)] += 1
    
    print("\nEdge counts by source and target type:")
    for (source_type, target_type), count in sorted(edge_types.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {source_type} -> {target_type}: {count}")
    
    # Check for isolated nodes
    isolated = list(nx.isolates(nx_graph))
    print(f"\nNumber of isolated nodes: {len(isolated)}")
    
    isolated_types = defaultdict(int)
    for node in isolated:
        node_type = nx_graph.nodes[node].get('type', 'unknown')
        isolated_types[node_type] += 1
    
    print("Isolated nodes by type:")
    for node_type, count in sorted(isolated_types.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {node_type}: {count}")
    
    return nx_graph, entities

In [ ]:
build_efficient_reference_graph(nkb)

Adding entities as nodes...
Added 131388 entity nodes to the graph
Building indexes...
Finding references through BNodes...


In [13]:
import rdflib
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter

# Load the RDF
g = rdflib.Graph()
g.parse("./mappings/NKB_RDF_V3.ttl", format="turtle")

# Basic statistics
print(f"Total number of triples: {len(g)}")

# Count entity types
entity_types = []
for s, p, o in g.triples((None, rdflib.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), None)):
    entity_types.append(str(o))

type_counts = Counter(entity_types)
print("\nEntity types and counts:")
for entity_type, count in type_counts.most_common():
    print(f"{entity_type}: {count}")

# Count predicate types
predicates = []
for s, p, o in g:
    predicates.append(str(p))

predicate_counts = Counter(predicates)
print("\nTop 20 predicates:")
for predicate, count in predicate_counts.most_common(20):
    print(f"{predicate}: {count}")

# Count BNodes (nodebags)
bnodes = sum(1 for s in g.subjects() if isinstance(s, rdflib.BNode))
print(f"\nNumber of BNodes (nodebags): {bnodes}")

Total number of triples: 1369675

Entity types and counts:
http://purl.bioontology.org/ontology/npo#NPO_1680: 83233
http://www.bioassayontology.org/bao#BAO_0000179: 24693
http://purl.obolibrary.org/obo/OBI_0000070: 22329
nan: 727
http://purl.bioontology.org/ontology/npo#NPO_199: 374
http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C63495: 302
http://purl.bioontology.org/ontology/npo#NPO_1853: 255
http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C48471: 129
http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C84280: 47
http://purl.enanomapper.org/onto/ENM_0000092: 21
http://purl.bioontology.org/ontology/npo#NPO_174: 16
http://purl.obolibrary.org/obo/ERO_0000833: 10

Top 20 predicates:
http://purl.org/dc/terms/identifier: 284602
http://purl.obolibrary.org/obo/OBI_0002110: 176420
http://www.w3.org/1999/02/22-rdf-syntax-ns#type: 132136
http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C42614: 130957
http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C68553: 112389
http://www.w3.org/199

In [32]:
g = rdflib.Graph()
g.parse("./mappings/NKB_RDF_V3.ttl", format="turtle")

<Graph identifier=Nafd87197d24e4fd4808eb8f0197a64e3 (<class 'rdflib.graph.Graph'>)>

In [36]:
def analyze_graph_structure(nx_graph, entities):
    """
    Analyze the graph structure to identify patterns and relationships.
    """
    import pandas as pd
    
    # Count nodes by type
    node_types = {}
    for node, data in nx_graph.nodes(data=True):
        node_type = data.get('type', 'unknown')
        node_types[node_type] = node_types.get(node_type, 0) + 1
    
    print("Node counts by type:")
    for node_type, count in sorted(node_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  {node_type}: {count}")
    
    # Analyze edge connections between different node types
    edge_types = {}
    for s, t, data in nx_graph.edges(data=True):
        source_type = nx_graph.nodes[s].get('type', 'unknown')
        target_type = nx_graph.nodes[t].get('type', 'unknown')
        relation = data.get('relation', 'unknown')
        
        key = (source_type, target_type, relation)
        edge_types[key] = edge_types.get(key, 0) + 1
    
    print("\nEdge counts by source type, target type, and relation:")
    edge_df = pd.DataFrame([
        {'source_type': s, 'target_type': t, 'relation': r, 'count': c}
        for (s, t, r), c in edge_types.items()
    ])
    
    # Summary by source and target type
    st_summary = edge_df.groupby(['source_type', 'target_type'])['count'].sum().reset_index()
    st_summary = st_summary.sort_values('count', ascending=False)
    
    print("\nEdge counts by source and target type:")
    for _, row in st_summary.iterrows():
        print(f"  {row['source_type']} -> {row['target_type']}: {row['count']}")
    
    # Identify isolated nodes
    isolated = list(nx.isolates(nx_graph))
    print(f"\nNumber of isolated nodes: {len(isolated)}")
    
    if isolated:
        isolated_types = {}
        for node in isolated:
            node_type = nx_graph.nodes[node].get('type', 'unknown')
            isolated_types[node_type] = isolated_types.get(node_type, 0) + 1
        
        print("Isolated nodes by type:")
        for node_type, count in sorted(isolated_types.items(), key=lambda x: x[1], reverse=True):
            print(f"  {node_type}: {count}")
    
    return edge_df

In [33]:

# Create the knowledge graph
nx_graph, entities, relation_counts = create_knowledge_graph_improved(g)

Found 10183 entities
Found 12031 edges with 2 different relation types


In [40]:
def create_complete_knowledge_graph(g):
    """
    Create a knowledge graph with explicit handling for each entity type.
    """
    import networkx as nx
    import rdflib
    from collections import defaultdict
    
    # Create a NetworkX graph
    nx_graph = nx.DiGraph()
    
    # Define entity types with their IRIs
    entity_iris = {
        'material': 'http://purl.bioontology.org/ontology/npo#NPO_199',
        'assay': 'http://purl.obolibrary.org/obo/OBI_0000070',
        'publication': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C48471',
        'medium': 'http://purl.bioontology.org/ontology/npo#NPO_1853',
        'result': 'http://www.bioassayontology.org/bao#BAO_0000179',
        'parameter': 'http://purl.bioontology.org/ontology/npo#NPO_1680',
        'materialfg': 'http://purl.bioontology.org/ontology/npo#NPO_174',
        'additive': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C63495',
        'contam': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C84280',
        'molecularresult': 'http://purl.obolibrary.org/obo/ERO_0000833'
    }
    
    # Find all entities and add them as nodes
    entities = {}
    entity_to_type = {}
    
    print("Identifying entities...")
    for entity_name, entity_iri in entity_iris.items():
        for s, p, o in g.triples((None, rdflib.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), 
                                  rdflib.URIRef(entity_iri))):
            if not isinstance(s, rdflib.BNode):  # Skip BNodes
                str_s = str(s)
                entities[str_s] = {'type': entity_name, 'properties': {}}
                entity_to_type[str_s] = entity_name
                nx_graph.add_node(str_s, type=entity_name)
    
    print(f"Found {len(entities)} entities")
    
    # Count number of each entity type
    type_counts = defaultdict(int)
    for _, data in entities.items():
        type_counts[data['type']] += 1
    
    print("Entity counts by type:")
    for entity_type, count in sorted(type_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {entity_type}: {count}")
    
    # Extract entity properties (direct literals)
    for entity_id in entities:
        for s, p, o in g.triples((rdflib.URIRef(entity_id), None, None)):
            if isinstance(o, rdflib.Literal):
                entities[entity_id]['properties'][str(p)] = str(o)
    
    # Create mappings from IDs to entities
    # This is critical for finding relationships
    id_predicates = {
        'material': 'http://purl.org/dc/terms/identifier',
        'assay': 'http://purl.org/dc/terms/identifier',
        'publication': 'http://purl.obolibrary.org/obo/OBI_0002110',  # DOI
        'medium': 'http://purl.org/dc/terms/identifier',
        'result': 'http://purl.org/dc/terms/identifier',
        'parameter': 'http://purl.org/dc/terms/identifier',
        'materialfg': 'http://purl.org/dc/terms/identifier',
        'additive': 'http://purl.org/dc/terms/identifier',
        'contam': 'http://purl.org/dc/terms/identifier',
        'molecularresult': 'http://purl.org/dc/terms/identifier'
    }
    
    # Create a mapping from ID to entity for each entity type
    id_to_entity = {entity_type: {} for entity_type in entity_iris.keys()}
    
    print("Building ID-to-entity mappings...")
    for entity_id, entity_data in entities.items():
        entity_type = entity_data['type']
        id_pred = id_predicates.get(entity_type)
        
        if id_pred:
            for s, p, o in g.triples((rdflib.URIRef(entity_id), rdflib.URIRef(id_pred), None)):
                id_value = str(o)
                id_to_entity[entity_type][id_value] = entity_id
    
    # Find direct entity-to-entity relationships
    print("Finding direct entity-to-entity relationships...")
    direct_edges = 0
    for s, p, o in g:
        if isinstance(s, rdflib.URIRef) and isinstance(o, rdflib.URIRef):
            str_s = str(s)
            str_o = str(o)
            
            if str_s in entities and str_o in entities and str_s != str_o:
                # Direct link between two entities
                s_type = entity_to_type.get(str_s)
                o_type = entity_to_type.get(str_o)
                nx_graph.add_edge(str_s, str_o, relation=str(p), s_type=s_type, o_type=o_type)
                direct_edges += 1
    
    print(f"Found {direct_edges} direct entity-to-entity relationships")
    
    # Find entity-to-ID references (foreign keys)
    print("Finding entity-to-ID references...")
    
    # These are the predicates used to reference other entities
    reference_predicates = [
        # Example: Material references Medium
        'http://purl.bioontology.org/ontology/npo#NPO_1853',  # medium
        # Assay references Material
        'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C93400',  # material
        # Result references Assay
        'http://purl.obolibrary.org/obo/OBI_0000070',  # assay
        # Parameter references Assay
        'http://purl.obolibrary.org/obo/OBI_0000070',  # assay
        # MaterialFG references Material
        'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C93400',  # material
        # Contam references Material
        'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C93400',  # material
        # Additive references Medium
        'http://purl.bioontology.org/ontology/npo#NPO_1853',  # medium
        # MolecularResult references Assay
        'http://purl.obolibrary.org/obo/OBI_0000070',  # assay
    ]
    
    fk_edges = 0
    for entity_id, entity_data in entities.items():
        entity_uri = rdflib.URIRef(entity_id)
        
        for ref_pred in reference_predicates:
            for s, p, o in g.triples((entity_uri, rdflib.URIRef(ref_pred), None)):
                if isinstance(o, rdflib.BNode):
                    # This is a reference through a BNode (nodebag)
                    # Find the ID in the BNode
                    bag = o
                    for bag_s, bag_p, bag_o in g.triples((bag, rdflib.URIRef('http://purl.org/dc/terms/identifier'), None)):
                        id_value = str(bag_o)
                        
                        # Determine the target entity type based on the reference predicate
                        target_type = None
                        if ref_pred == 'http://purl.bioontology.org/ontology/npo#NPO_1853':
                            target_type = 'medium'
                        elif ref_pred == 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C93400':
                            target_type = 'material'
                        elif ref_pred == 'http://purl.obolibrary.org/obo/OBI_0000070':
                            target_type = 'assay'
                        
                        if target_type and id_value in id_to_entity[target_type]:
                            target_id = id_to_entity[target_type][id_value]
                            relation = f"references_{target_type}"
                            nx_graph.add_edge(entity_id, target_id, relation=relation)
                            fk_edges += 1
    
    print(f"Found {fk_edges} foreign key relationships")
    
    # Process publication references (DOIs)
    print("Finding publication references...")
    doi_pred = 'http://purl.obolibrary.org/obo/OBI_0002110'  # DOI predicate
    pub_edges = 0
    
    # First, collect all publication DOIs
    publication_dois = {}
    for entity_id, entity_data in entities.items():
        if entity_data['type'] == 'publication':
            # Find the DOI of this publication
            for s, p, o in g.triples((rdflib.URIRef(entity_id), rdflib.URIRef(doi_pred), None)):
                doi = str(o)
                publication_dois[doi] = entity_id
    
    # Then find all entities that reference publications
    for entity_id, entity_data in entities.items():
        if entity_data['type'] != 'publication':  # Skip publications themselves
            entity_uri = rdflib.URIRef(entity_id)
            
            # Direct DOI reference
            for s, p, o in g.triples((entity_uri, rdflib.URIRef(doi_pred), None)):
                doi = str(o)
                if doi in publication_dois:
                    pub_id = publication_dois[doi]
                    nx_graph.add_edge(entity_id, pub_id, relation="references_publication")
                    pub_edges += 1
            
            # DOI reference through a BNode
            for s, p, o in g.triples((entity_uri, None, None)):
                if isinstance(o, rdflib.BNode):
                    # Check if this BNode contains a DOI
                    for bag_s, bag_p, bag_o in g.triples((o, rdflib.URIRef(doi_pred), None)):
                        doi = str(bag_o)
                        if doi in publication_dois:
                            pub_id = publication_dois[doi]
                            nx_graph.add_edge(entity_id, pub_id, relation="references_publication_via_bag")
                            pub_edges += 1
            
            # Special case: publication_DOI column 
            for s, p, o in g.triples((entity_uri, rdflib.Literal('publication_DOI'), None)):
                doi = str(o)
                if doi in publication_dois:
                    pub_id = publication_dois[doi]
                    nx_graph.add_edge(entity_id, pub_id, relation="references_publication_direct")
                    pub_edges += 1
    
    print(f"Found {pub_edges} publication reference relationships")
    
    # Handle specific entity type relationships that might be missed
    
    # Material to Medium relationships
    print("Finding Material-Medium relationships...")
    material_medium_edges = 0
    
    for entity_id, entity_data in entities.items():
        if entity_data['type'] == 'material':
            entity_uri = rdflib.URIRef(entity_id)
            
            # Check for medium references
            for s, p, o in g.triples((entity_uri, rdflib.Literal('medium_MediumID'), None)):
                medium_id = str(o)
                for medium_entity, medium_data in entities.items():
                    if medium_data['type'] == 'medium':
                        # Check if this medium has the matching ID
                        medium_uri = rdflib.URIRef(medium_entity)
                        for m_s, m_p, m_o in g.triples((medium_uri, rdflib.URIRef(id_predicates['medium']), None)):
                            if str(m_o) == medium_id:
                                nx_graph.add_edge(entity_id, medium_entity, relation="uses_medium")
                                material_medium_edges += 1
    
    print(f"Found {material_medium_edges} Material-Medium relationships")
    
    # Process Result to Parameter relationships
    print("Finding Result-Parameter relationships...")
    result_param_edges = 0
    
    for entity_id, entity_data in entities.items():
        if entity_data['type'] == 'result':
            # Results should be connected to their parameters
            # They usually share the same assay
            result_assay = None
            
            # Find the assay for this result
            for s, p, o in g.triples((rdflib.URIRef(entity_id), rdflib.URIRef('http://purl.obolibrary.org/obo/OBI_0000070'), None)):
                if isinstance(o, rdflib.BNode):
                    for bag_s, bag_p, bag_o in g.triples((o, rdflib.URIRef('http://purl.org/dc/terms/identifier'), None)):
                        assay_id = str(bag_o)
                        result_assay = assay_id
            
            if result_assay:
                # Find parameters with the same assay
                for param_id, param_data in entities.items():
                    if param_data['type'] == 'parameter':
                        param_assay = None
                        
                        # Find the assay for this parameter
                        for s, p, o in g.triples((rdflib.URIRef(param_id), rdflib.URIRef('http://purl.obolibrary.org/obo/OBI_0000070'), None)):
                            if isinstance(o, rdflib.BNode):
                                for bag_s, bag_p, bag_o in g.triples((o, rdflib.URIRef('http://purl.org/dc/terms/identifier'), None)):
                                    param_assay_id = str(bag_o)
                                    param_assay = param_assay_id
                        
                        if param_assay == result_assay:
                            nx_graph.add_edge(entity_id, param_id, relation="has_parameter")
                            result_param_edges += 1
    
    print(f"Found {result_param_edges} Result-Parameter relationships")
    
    # Process Material to Material Functional Group relationships
    print("Finding Material-MaterialFG relationships...")
    material_fg_edges = 0
    
    for entity_id, entity_data in entities.items():
        if entity_data['type'] == 'materialfg':
            # Find the material it's connected to
            for s, p, o in g.triples((rdflib.URIRef(entity_id), rdflib.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C93400'), None)):
                if isinstance(o, rdflib.BNode):
                    for bag_s, bag_p, bag_o in g.triples((o, rdflib.URIRef('http://purl.org/dc/terms/identifier'), None)):
                        material_id = str(bag_o)
                        
                        # Find the material with this ID
                        if material_id in id_to_entity['material']:
                            target_id = id_to_entity['material'][material_id]
                            nx_graph.add_edge(entity_id, target_id, relation="belongs_to_material")
                            material_fg_edges += 1
    
    print(f"Found {material_fg_edges} Material-MaterialFG relationships")
    
    # Check for isolated nodes
    isolated = list(nx.isolates(nx_graph))
    print(f"\nNumber of isolated nodes: {len(isolated)}")
    
    if isolated:
        isolated_types = defaultdict(int)
        for node in isolated:
            node_type = nx_graph.nodes[node].get('type', 'unknown')
            isolated_types[node_type] += 1
        
        print("Isolated nodes by type:")
        for node_type, count in sorted(isolated_types.items(), key=lambda x: x[1], reverse=True):
            print(f"  {node_type}: {count}")
    
    # Calculate final edge statistics
    edge_types = defaultdict(int)
    for s, t, data in nx_graph.edges(data=True):
        source_type = nx_graph.nodes[s].get('type', 'unknown')
        target_type = nx_graph.nodes[t].get('type', 'unknown')
        edge_types[(source_type, target_type)] += 1
    
    print("\nEdge counts by source and target type:")
    for (source_type, target_type), count in sorted(edge_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  {source_type} -> {target_type}: {count}")
    
    return nx_graph, entities

In [41]:
create_complete_knowledge_graph(g)

Identifying entities...
Found 10183 entities
Entity counts by type:
  assay: 9881
  additive: 302
Building ID-to-entity mappings...
Finding direct entity-to-entity relationships...
Found 0 direct entity-to-entity relationships
Finding entity-to-ID references...
Found 0 foreign key relationships
Finding publication references...
Found 0 publication reference relationships
Finding Material-Medium relationships...
Found 0 Material-Medium relationships
Finding Result-Parameter relationships...
Found 0 Result-Parameter relationships
Finding Material-MaterialFG relationships...
Found 0 Material-MaterialFG relationships

Number of isolated nodes: 10183
Isolated nodes by type:
  assay: 9881
  additive: 302

Edge counts by source and target type:


(<networkx.classes.digraph.DiGraph at 0x410933d50>,
 {'http://example.org/assay/0': {'type': 'assay',
   'properties': {'http://purl.obolibrary.org/obo/OBI_0002110': '10.1002/cyto.22342',
    'http://purl.org/dc/terms/identifier': '1'}},
  'http://example.org/assay/1': {'type': 'assay',
   'properties': {'http://purl.obolibrary.org/obo/OBI_0002110': '10.1002/cyto.a.20927',
    'http://purl.org/dc/terms/identifier': '1'}},
  'http://example.org/assay/10': {'type': 'assay',
   'properties': {'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C25372': 'physical characterization',
    'http://purl.obolibrary.org/obo/OBI_0002110': '10.1002/etc.2583',
    'http://purl.org/dc/terms/identifier': '1'}},
  'http://example.org/assay/100': {'type': 'assay',
   'properties': {'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C25372': 'chemical characterization',
    'http://purl.obolibrary.org/obo/OBI_0002110': '10.1021/es902240k',
    'http://purl.org/dc/terms/identifier': '1'}},
  'http://exam

In [42]:
def analyze_rdf_structure(g):
    """
    Deeply analyze the RDF structure to understand how entities are connected.
    """
    import rdflib
    from collections import defaultdict
    
    # Count all subjects, predicates, and objects
    subjects = defaultdict(int)
    predicates = defaultdict(int)
    objects = defaultdict(int)
    
    for s, p, o in g:
        subjects[type(s).__name__] += 1
        predicates[str(p)] += 1
        objects[type(o).__name__] += 1
    
    print("Subject types:")
    for s_type, count in sorted(subjects.items(), key=lambda x: x[1], reverse=True):
        print(f"  {s_type}: {count}")
    
    print("\nObject types:")
    for o_type, count in sorted(objects.items(), key=lambda x: x[1], reverse=True):
        print(f"  {o_type}: {count}")
    
    # Count BNode connections
    bnode_connections = defaultdict(int)
    for s, p, o in g:
        if isinstance(s, rdflib.URIRef) and isinstance(o, rdflib.BNode):
            bnode_connections["URI -> BNode"] += 1
        elif isinstance(s, rdflib.BNode) and isinstance(o, rdflib.URIRef):
            bnode_connections["BNode -> URI"] += 1
        elif isinstance(s, rdflib.BNode) and isinstance(o, rdflib.Literal):
            bnode_connections["BNode -> Literal"] += 1
        elif isinstance(s, rdflib.BNode) and isinstance(o, rdflib.BNode):
            bnode_connections["BNode -> BNode"] += 1
    
    print("\nBNode connection patterns:")
    for pattern, count in sorted(bnode_connections.items(), key=lambda x: x[1], reverse=True):
        print(f"  {pattern}: {count}")
    
    # Find how entities are typed
    type_predicate = rdflib.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type')
    entity_types = defaultdict(int)
    
    for s, p, o in g.triples((None, type_predicate, None)):
        if isinstance(s, rdflib.URIRef):
            entity_types[str(o)] += 1
    
    print("\nEntity types by RDF:type predicate:")
    for entity_type, count in sorted(entity_types.items(), key=lambda x: x[1], reverse=True)[:15]:
        print(f"  {entity_type}: {count}")
    
    # Analyze common predicates used with BNodes
    bnode_predicates = defaultdict(int)
    for s, p, o in g:
        if isinstance(s, rdflib.URIRef) and isinstance(o, rdflib.BNode):
            bnode_predicates[str(p)] += 1
    
    print("\nTop predicates connecting URIs to BNodes:")
    for pred, count in sorted(bnode_predicates.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {pred}: {count}")
    
    # Find what's inside the BNodes
    # Sample a few BNodes to understand their structure
    bnode_sample = []
    for s, p, o in g:
        if isinstance(o, rdflib.BNode) and isinstance(s, rdflib.URIRef):
            bnode = o
            contents = []
            
            # Get all properties of this BNode
            for bs, bp, bo in g.triples((bnode, None, None)):
                contents.append((str(bp), str(type(bo).__name__), str(bo)[:50] if len(str(bo)) > 50 else str(bo)))
            
            bnode_sample.append((str(s), str(p), contents))
            
            if len(bnode_sample) >= 5:
                break
    
    print("\nSample BNode structures:")
    for s, p, contents in bnode_sample:
        print(f"\nURI {s} connected to BNode via {p}")
        for bp, bo_type, bo in contents:
            print(f"  - {bp} -> ({bo_type}) {bo}")
    
    return subjects, predicates, objects, bnode_connections, entity_types

In [43]:
analyze_rdf_structure(g)

Subject types:
  URIRef: 70989
  BNode: 40127

Object types:
  Literal: 71179
  BNode: 20063
  URIRef: 19874

BNode connection patterns:
  BNode -> Literal: 40127
  URI -> BNode: 20063

Entity types by RDF:type predicate:
  http://purl.obolibrary.org/obo/OBI_0000070: 9881
  http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C63495: 302

Top predicates connecting URIs to BNodes:
  http://purl.bioontology.org/ontology/npo#NPO_1853: 10182
  http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C93400: 9881

Sample BNode structures:

URI http://example.org/assay/1134 connected to BNode via http://purl.bioontology.org/ontology/npo#NPO_1853
  - http://purl.obolibrary.org/obo/OBI_0002110 -> (Literal) 10.3109/17435390.2010.539711
  - http://purl.org/dc/terms/identifier -> (Literal) 1

URI http://example.org/assay/14067 connected to BNode via http://purl.bioontology.org/ontology/npo#NPO_1853
  - http://purl.obolibrary.org/obo/OBI_0002110 -> (Literal) 10.1016/j.colsurfa.2013.11.004
  - http://purl.

(defaultdict(int, {'BNode': 40127, 'URIRef': 70989}),
 defaultdict(int,
             {'http://purl.org/dc/terms/identifier': 30245,
              'http://purl.bioontology.org/ontology/npo#NPO_1853': 10182,
              'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C42614': 10183,
              'http://purl.obolibrary.org/obo/OBI_0002110': 29944,
              'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C25372': 9881,
              'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C93400': 9881,
              'http://www.w3.org/1999/02/22-rdf-syntax-ns#type': 10183,
              'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C68553': 302,
              'http://www.w3.org/1999/02/22-rdf-syntax-ns#value': 302,
              'http://www.w3.org/2000/01/rdf-schema#label': 7,
              'http://www.w3.org/2000/01/rdf-schema#subClassOf': 6}),
 defaultdict(int, {'Literal': 71179, 'BNode': 20063, 'URIRef': 19874}),
 defaultdict(int, {'BNode -> Literal': 40127, 'URI -> B

In [45]:
def extract_nodebag_connections(g):
    """
    Extract connections through nodebags (BNodes) in the RDF.
    """
    import rdflib
    import networkx as nx
    from collections import defaultdict
    
    # Create a NetworkX graph
    nx_graph = nx.DiGraph()
    
    # Define entity types with their IRIs
    entity_iris = {
        'material': 'http://purl.bioontology.org/ontology/npo#NPO_199',
        'assay': 'http://purl.obolibrary.org/obo/OBI_0000070',
        'publication': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C48471',
        'medium': 'http://purl.bioontology.org/ontology/npo#NPO_1853',
        'result': 'http://www.bioassayontology.org/bao#BAO_0000179',
        'parameter': 'http://purl.bioontology.org/ontology/npo#NPO_1680',
        'materialfg': 'http://purl.bioontology.org/ontology/npo#NPO_174',
        'additive': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C63495',
        'contam': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C84280',
        'molecularresult': 'http://purl.obolibrary.org/obo/ERO_0000833'
    }
    
    # First, identify all entities and add them as nodes
    entities = {}
    entity_uris = {}
    
    for entity_name, entity_iri in entity_iris.items():
        entity_type_uri = rdflib.URIRef(entity_iri)
        for s, p, o in g.triples((None, rdflib.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), entity_type_uri)):
            if isinstance(s, rdflib.URIRef):
                str_s = str(s)
                entities[str_s] = {'type': entity_name}
                entity_uris[str_s] = s
                nx_graph.add_node(str_s, type=entity_name)
    
    print(f"Found {len(entities)} entities")
    
    # Create a mapping of BNodes to the entities they're connected to
    bnode_to_entity = {}
    entity_to_bnodes = defaultdict(list)
    
    # Find all BNodes connected directly to entities
    for entity_uri_str, entity_data in entities.items():
        entity_uri = entity_uris[entity_uri_str]
        
        for s, p, o in g.triples((entity_uri, None, None)):
            if isinstance(o, rdflib.BNode):
                bnode_str = str(o)
                bnode_to_entity[bnode_str] = entity_uri_str
                entity_to_bnodes[entity_uri_str].append((bnode_str, str(p)))
    
    print(f"Found {len(bnode_to_entity)} BNodes connected to entities")
    
    # Find all DOI references
    doi_predicate = rdflib.URIRef('http://purl.obolibrary.org/obo/OBI_0002110')
    id_predicate = rdflib.URIRef('http://purl.org/dc/terms/identifier')
    
    # Map from DOI to publication entity
    doi_to_publication = {}
    for entity_uri_str, entity_data in entities.items():
        if entity_data['type'] == 'publication':
            entity_uri = entity_uris[entity_uri_str]
            for s, p, o in g.triples((entity_uri, doi_predicate, None)):
                if isinstance(o, rdflib.Literal):
                    doi_to_publication[str(o)] = entity_uri_str
    
    print(f"Found {len(doi_to_publication)} publications with DOIs")
    
    # Find connections through DOIs
    doi_connections = 0
    for entity_uri_str, entity_data in entities.items():
        if entity_data['type'] != 'publication':
            entity_uri = entity_uris[entity_uri_str]
            
            # Direct DOI reference
            for s, p, o in g.triples((entity_uri, doi_predicate, None)):
                if isinstance(o, rdflib.Literal) and str(o) in doi_to_publication:
                    publication_uri = doi_to_publication[str(o)]
                    nx_graph.add_edge(entity_uri_str, publication_uri, relation='references_publication_doi')
                    doi_connections += 1
            
            # DOI reference through BNodes
            for bnode_str, predicate in entity_to_bnodes[entity_uri_str]:
                bnode = rdflib.BNode(bnode_str.split('N')[-1])
                for bs, bp, bo in g.triples((bnode, doi_predicate, None)):
                    if isinstance(bo, rdflib.Literal) and str(bo) in doi_to_publication:
                        publication_uri = doi_to_publication[str(bo)]
                        nx_graph.add_edge(entity_uri_str, publication_uri, relation=f'references_publication_via_{predicate}')
                        doi_connections += 1
    
    print(f"Found {doi_connections} connections through DOIs")
    
    # Find ID-based connections
    id_connections = 0
    
    # Map from ID to entity
    id_to_entity = defaultdict(dict)
    for entity_uri_str, entity_data in entities.items():
        entity_type = entity_data['type']
        entity_uri = entity_uris[entity_uri_str]
        
        for s, p, o in g.triples((entity_uri, id_predicate, None)):
            if isinstance(o, rdflib.Literal):
                id_to_entity[entity_type][str(o)] = entity_uri_str
    
    # Find connections through IDs in BNodes
    for bnode_str, source_entity in bnode_to_entity.items():
        bnode = rdflib.BNode(bnode_str.split('N')[-1])
        source_type = entities[source_entity]['type']
        
        # The predicate connecting the entity to the BNode indicates the target entity type
        for bs, bp, bo in g.triples((bnode, id_predicate, None)):
            if isinstance(bo, rdflib.Literal):
                id_value = str(bo)
                
                # Try to find this ID in other entity types
                for target_type, id_map in id_to_entity.items():
                    if target_type != source_type and id_value in id_map:
                        target_entity = id_map[id_value]
                        predicate = entity_to_bnodes[source_entity][0][1] if entity_to_bnodes[source_entity] else "unknown"
                        nx_graph.add_edge(source_entity, target_entity, relation=f'references_{target_type}_via_{predicate}')
                        id_connections += 1
    
    print(f"Found {id_connections} connections through IDs")
    
    # Find connections between entities of the same type (e.g., assay -> assay)
    same_type_connections = 0
    
    # Look at column names stored as Literals
    # This is a special case in the EPA NaKnowBase RDF structure
    column_connections = defaultdict(int)
    
    for s, p, o in g:
        if isinstance(s, rdflib.URIRef) and isinstance(p, rdflib.Literal) and isinstance(o, rdflib.Literal):
            # This is likely a column reference like "material_MaterialID"
            if '_' in str(p):
                parts = str(p).split('_')
                if len(parts) >= 2:
                    ref_type = parts[0].lower()  # e.g., "material" from "material_MaterialID"
                    
                    if ref_type in entity_iris:
                        # This is a reference to another entity
                        source_uri_str = str(s)
                        if source_uri_str in entities:
                            id_value = str(o)
                            if ref_type in id_to_entity and id_value in id_to_entity[ref_type]:
                                target_uri_str = id_to_entity[ref_type][id_value]
                                nx_graph.add_edge(source_uri_str, target_uri_str, relation=f'direct_column_reference_to_{ref_type}')
                                column_connections[ref_type] += 1
                                same_type_connections += 1
    
    print(f"Found {same_type_connections} connections through column references")
    for ref_type, count in column_connections.items():
        print(f"  References to {ref_type}: {count}")
    
    # Count final edge statistics
    edge_types = defaultdict(int)
    for s, t, data in nx_graph.edges(data=True):
        source_type = nx_graph.nodes[s].get('type', 'unknown')
        target_type = nx_graph.nodes[t].get('type', 'unknown')
        edge_types[(source_type, target_type)] += 1
    
    print("\nEdge counts by source and target type:")
    for (source_type, target_type), count in sorted(edge_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  {source_type} -> {target_type}: {count}")
    
    # Check for isolated nodes
    isolated = list(nx.isolates(nx_graph))
    print(f"\nNumber of isolated nodes: {len(isolated)}")
    
    if isolated:
        isolated_types = defaultdict(int)
        for node in isolated:
            node_type = nx_graph.nodes[node].get('type', 'unknown')
            isolated_types[node_type] += 1
        
        print("Isolated nodes by type:")
        for node_type, count in sorted(isolated_types.items(), key=lambda x: x[1], reverse=True):
            print(f"  {node_type}: {count}")
    
    return nx_graph, entities

In [47]:
def sample_rdf_data(g, n=100):
    """
    Sample a few triples from the RDF to understand its structure.
    """
    import random
    
    # Convert to list for sampling
    triples = list(g)
    
    # Sample n triples
    if len(triples) > n:
        sample = random.sample(triples, n)
    else:
        sample = triples
    
    for s, p, o in sample:
        print(f"{type(s).__name__}: {s}")
        print(f"  -- {type(p).__name__}: {p}")
        print(f"  --> {type(o).__name__}: {o}")
        print()

    # Try to find examples of different connection patterns
    patterns = {
        "URI->URI": [],
        "URI->BNode": [],
        "BNode->URI": [],
        "URI->Literal (column name)": [],
        "BNode->Literal (ID)": []
    }
    
    # Find examples matching each pattern
    for s, p, o in g:
        if len(patterns["URI->URI"]) < 5 and isinstance(s, rdflib.URIRef) and isinstance(o, rdflib.URIRef):
            patterns["URI->URI"].append((s, p, o))
        elif len(patterns["URI->BNode"]) < 5 and isinstance(s, rdflib.URIRef) and isinstance(o, rdflib.BNode):
            patterns["URI->BNode"].append((s, p, o))
        elif len(patterns["BNode->URI"]) < 5 and isinstance(s, rdflib.BNode) and isinstance(o, rdflib.URIRef):
            patterns["BNode->URI"].append((s, p, o))
        elif len(patterns["URI->Literal (column name)"]) < 5 and isinstance(s, rdflib.URIRef) and isinstance(p, rdflib.Literal):
            patterns["URI->Literal (column name)"].append((s, p, o))
        elif len(patterns["BNode->Literal (ID)"]) < 5 and isinstance(s, rdflib.BNode) and isinstance(o, rdflib.Literal) and str(p).endswith("identifier"):
            patterns["BNode->Literal (ID)"].append((s, p, o))
    
    # Print examples of each pattern
    print("\nExample connection patterns:")
    for pattern_name, examples in patterns.items():
        print(f"\n{pattern_name} examples:")
        for i, (s, p, o) in enumerate(examples):
            print(f"Example {i+1}:")
            print(f"  Subject: {s}")
            print(f"  Predicate: {p}")
            print(f"  Object: {o}")
    
    return patterns

In [48]:
sample_rdf_data(g)

URIRef: http://example.org/assay/16025
  -- URIRef: http://purl.bioontology.org/ontology/npo#NPO_1853
  --> BNode: nd9e43a5bafbd4dd585b1deaec7d4b0b4b13700

URIRef: http://example.org/assay/17647
  -- URIRef: http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C25372
  --> Literal: life cycle

URIRef: http://example.org/assay/15127
  -- URIRef: http://purl.obolibrary.org/obo/OBI_0002110
  --> Literal: 10.1016/j.scitotenv.2016.11.029

URIRef: http://example.org/assay/18762
  -- URIRef: http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C25372
  --> Literal: in vitro 

BNode: nd9e43a5bafbd4dd585b1deaec7d4b0b4b5991
  -- URIRef: http://purl.obolibrary.org/obo/OBI_0002110
  --> Literal: 10.1016/j.watres.2012.10.026

URIRef: http://example.org/assay/15643
  -- URIRef: http://purl.bioontology.org/ontology/npo#NPO_1853
  --> BNode: nd9e43a5bafbd4dd585b1deaec7d4b0b4b12850

URIRef: http://example.org/assay/10983
  -- URIRef: http://purl.bioontology.org/ontology/npo#NPO_1853
  --> BNode: nd9e43a5ba

{'URI->URI': [(rdflib.term.URIRef('http://example.org/assay/13805'),
   rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C42614'),
   rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C43379')),
  (rdflib.term.URIRef('http://example.org/assay/15191'),
   rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C42614'),
   rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C43379')),
  (rdflib.term.URIRef('http://example.org/assay/18439'),
   rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
   rdflib.term.URIRef('http://purl.obolibrary.org/obo/OBI_0000070')),
  (rdflib.term.URIRef('http://example.org/assay/16584'),
   rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
   rdflib.term.URIRef('http://purl.obolibrary.org/obo/OBI_0000070')),
  (rdflib.term.URIRef('http://example.org/assay/17837'),
   rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C42

In [46]:
extract_nodebag_connections(g)

Found 10183 entities
Found 20063 BNodes connected to entities
Found 0 publications with DOIs
Found 0 connections through DOIs
Found 207 connections through IDs
Found 0 connections through column references

Edge counts by source and target type:
  additive -> assay: 207

Number of isolated nodes: 9971
Isolated nodes by type:
  assay: 9876
  additive: 95


(<networkx.classes.digraph.DiGraph at 0x3e79b1050>,
 {'http://example.org/assay/0': {'type': 'assay'},
  'http://example.org/assay/1': {'type': 'assay'},
  'http://example.org/assay/10': {'type': 'assay'},
  'http://example.org/assay/100': {'type': 'assay'},
  'http://example.org/assay/1000': {'type': 'assay'},
  'http://example.org/assay/10000': {'type': 'assay'},
  'http://example.org/assay/10001': {'type': 'assay'},
  'http://example.org/assay/10002': {'type': 'assay'},
  'http://example.org/assay/10003': {'type': 'assay'},
  'http://example.org/assay/10004': {'type': 'assay'},
  'http://example.org/assay/10005': {'type': 'assay'},
  'http://example.org/assay/10006': {'type': 'assay'},
  'http://example.org/assay/10007': {'type': 'assay'},
  'http://example.org/assay/10008': {'type': 'assay'},
  'http://example.org/assay/10009': {'type': 'assay'},
  'http://example.org/assay/1001': {'type': 'assay'},
  'http://example.org/assay/10010': {'type': 'assay'},
  'http://example.org/assay/

In [38]:
analyze_graph_structure(nx_graph, entities)

Node counts by type:
  assay: 9881
  additive: 302

Edge counts by source type, target type, and relation:

Edge counts by source and target type:
  assay -> assay: 11824
  additive -> assay: 207

Number of isolated nodes: 253
Isolated nodes by type:
  assay: 158
  additive: 95


,source_type,target_type,relation,count
0,assay,assay,http://purl.bioontology.org/ontology/npo#NPO_1...,6834
1,assay,assay,http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus...,4990
2,additive,assay,http://purl.bioontology.org/ontology/npo#NPO_1...,207


In [50]:
create_reference_based_graph(g)

Found 10183 entities
Created 5043820 edges based on shared DOI references
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Users/pranavsingh/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3505, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/ps/cxk7gpds70390sk1wh4zjd240000gn/T/ipykernel_9889/4230367256.py", line 1, in <module>
    create_reference_based_graph(g)
  File "/var/folders/ps/cxk7gpds70390sk1wh4zjd240000gn/T/ipykernel_9889/3729011372.py", line -1, in create_reference_based_graph
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/pranavsingh/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 2102, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pranavsingh/anaconda3/lib/python3.11/site-packages/IPython/core/ultratb.py", line 1310, in structured_traceback
    return FormattedTB.structured

In [22]:
def build_gnn_for_reference_graph(nx_graph, entities):
    """
    Build a Graph Neural Network for a graph based on shared references.
    """
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch_geometric.nn import GCNConv, SAGEConv
    from torch_geometric.data import Data
    import numpy as np
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    
    # Create node features
    entity_types = sorted(set(data['type'] for data in entities.values()))
    type_encoder = OneHotEncoder(sparse=False)
    type_encoder.fit([[t] for t in entity_types])
    
    # Create node mapping
    node_mapping = {node: i for i, node in enumerate(nx_graph.nodes())}
    
    # Create node features
    node_features = {}
    for node_id in nx_graph.nodes():
        # One-hot encode the node type
        node_type = nx_graph.nodes[node_id].get('type', 'unknown')
        type_feature = type_encoder.transform([[node_type]])[0]
        
        # Add graph structure features
        in_degree = nx_graph.in_degree(node_id)
        out_degree = nx_graph.out_degree(node_id)
        
        # Combine features
        node_features[node_id] = np.concatenate([type_feature, [in_degree, out_degree]])
    
    # Create edge index
    edge_index = []
    for u, v in nx_graph.edges():
        edge_index.append([node_mapping[u], node_mapping[v]])
    
    # Convert to PyTorch tensors
    x = torch.tensor([node_features[node] for node in nx_graph.nodes()], dtype=torch.float)
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    
    # Create PyTorch Geometric Data object
    data = Data(x=x, edge_index=edge_index)
    
    # Define GNN model
    class GNN(nn.Module):
        def __init__(self, in_channels, hidden_channels, out_channels):
            super(GNN, self).__init__()
            self.conv1 = GCNConv(in_channels, hidden_channels)
            self.conv2 = GCNConv(hidden_channels, out_channels)
        
        def forward(self, x, edge_index):
            x = self.conv1(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=0.5, training=self.training)
            x = self.conv2(x, edge_index)
            return x
        
        def decode(self, z, edge_index):
            # Inner product between node embeddings to predict links
            src, dst = edge_index
            return torch.sum(z[src] * z[dst], dim=1)
    
    # Initialize model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = GNN(x.shape[1], 64, 32).to(device)
    data = data.to(device)
    
    # Initialize optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    # Train model
    model.train()
    for epoch in range(200):
        optimizer.zero_grad()
        z = model(data.x, data.edge_index)
        loss = F.binary_cross_entropy_with_logits(
            model.decode(z, data.edge_index),
            torch.ones(data.edge_index.size(1)).to(device)
        )
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")
    
    return model, node_mapping, data